# Imports and Loads

In [ ]:
TRAIN_PATH = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\train.csv'
TEST_PATH = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\test.csv'

OOF_CAT = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\models\oof_cat_new.npy'
OOF_LGB = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\models\oof_lightgbm.npy'
OOF_XGB = r''

TEST_CAT = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\models\test_cat_new.npy'
TEST_LGB = r'C:\Users\harwi\Downloads\student_health_risk_prediction_system\models\test_lightgbm.npy'
TEST_XGB = r''

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score
import optuna

train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)

oof_cat= np.load(OOF_CAT)
oof_lgb= np.load(OOF_LGB)
oof_xgb = np.load(OOF_XGB)

test_cat= np.load(TEST_CAT)
test_lgb= np.load(TEST_LGB)
test_xgb = np.load(TEST_XGB)


In [ ]:
OOF = {
    "cat": oof_cat,
    "lgb": oof_lgb,
    "xgb": oof_xgb,
}

TEST = {
    "cat": test_cat,
    "lgb": test_lgb,
    "xgb": test_xgb,
}

MODELS = {
    "CatBoost": oof_cat,
    "LightGBM": oof_lgb,
    "XGBoost": oof_xgb,
}

In [ ]:
lb = LabelEncoder()
y_kag = lb.fit_transform(train['health_condition'])

# Analysising the models

In [ ]:
predictions = {}

for name, probs in MODELS.items():
    predictions[name] = probs.argmax(axis=1)

model_names = list(predictions.keys())

agreement = pd.DataFrame(
    index=model_names,
    columns=model_names,
    dtype=float
)

In [ ]:
for model1 in model_names:

    for model2 in model_names:

        same_prediction = (
            predictions[model1]
            ==
            predictions[model2]
        )

        agreement.loc[model1, model2] = same_prediction.mean()

In [ ]:
agreement.head()

In [ ]:
plt.figure(figsize=(7,6))

sns.heatmap(
    agreement,
    annot=True,
    cmap="RdYlGn",
    fmt=".3f"
)

plt.title("Model Agreement")
plt.show()

In [ ]:
distribution = pd.DataFrame()

for name in model_names:

    pred = predictions[name]

    percent = (
        pd.Series(pred)
        .value_counts(normalize=True)
        .sort_index()
    )

    distribution[name] = percent

In [ ]:
distribution = distribution.T
print(distribution)

In [ ]:
distribution.plot(
    kind="bar",
    stacked=True,
    figsize=(8,5)
)

plt.ylabel("Percentage")
plt.title("Predicted Class Distribution")
plt.show()

**Observation**- According to the above analysis we can conclude our models are not too much diverse. Therefore there are very  changes that we do not get too much benefit from blending.

### Using Arithmetic Mean for Blending

In [ ]:
weights = {
    "cat": 0.50,
    "lgb": 0.30,
    "xgb": 0.20,
}

In [ ]:
blend_oof = np.zeros_like(oof_cat)

for name in weights:
    blend_oof += weights[name] * OOF[name]

pred = blend_oof.argmax(axis=1)

score = balanced_accuracy_score(
    y_kag,
    pred
)

print(score)

In [ ]:
blend_test = np.zeros_like(test_cat)

for name in weights:
    blend_test += weights[name] * TEST[name]

submission = train['id'].copy()

pred = blend_test.argmax(axis=1)

submission["health_condition"] = lb.inverse_transform(pred)

submission.to_csv(
    "submission_blend_gm.csv",
    index=False
)

## Geometric mean


In [ ]:
eps = 1e-15

blend_oof = np.zeros_like(oof_cat)

for name in weights:
    blend_oof += weights[name] * np.log(np.clip(OOF[name], eps, 1.0))

blend_oof = np.exp(blend_oof)

# Normalize so each row sums to 1
blend_oof = blend_oof / blend_oof.sum(axis=1, keepdims=True)

pred = blend_oof.argmax(axis=1)

score = balanced_accuracy_score(
    y_kag,
    pred
)

print(score)

eps = 1e-15

blend_test = np.zeros_like(test_cat)

for name in weights:
    blend_test += weights[name] * np.log(np.clip(TEST[name], eps, 1.0))

blend_test = np.exp(blend_test)

blend_test = blend_test / blend_test.sum(axis=1, keepdims=True)

pred = blend_test.argmax(axis=1)

submission.drop(columns = ['health_condition'],inplace = True)
submission["health_condition"] = lb.inverse_transform(pred)

submission.to_csv(
    "submission_blend_gm.csv",
    index=False
)

# Automatic Weight Imputation using optuna for arithmetic mean

In [ ]:
def objective(trial):

    weights = {}

    for model in OOF.keys():
        weights[model] = trial.suggest_float(
            model,
            0.0,
            1.0
        )

    # Normalize weights
    total = sum(weights.values())

    weights = {
        k: v / total
        for k, v in weights.items()
    }

    blend = np.zeros_like(next(iter(OOF.values())))

    for model in OOF.keys():
        blend += weights[model] * OOF[model]

    pred = blend.argmax(axis=1)

    score = balanced_accuracy_score(
        y_kag,
        pred
    )

    return score

In [ ]:
study = optuna.create_study(
    direction="maximize",
    study_name="blend_weights",
    storage="sqlite:///blend.db",
    load_if_exists=True
)

study.optimize(
    objective,
    n_trials=300,
    show_progress_bar=True
)